# IEEE-CIS: EDA legacy preprocessing + 13 research models

Notebook này chỉ dùng bộ dữ liệu Kaggle `ieee-fraud-detection`. Phần tạo đặc trưng bám theo `eda-and-models.ipynb`: LEFT JOIN transaction–identity, thống kê nhóm, tách email, lọc cột theo ngưỡng 90%, Label Encoding trên train+test, sắp train theo `TransactionDT`, rồi KFold 5 block không shuffle.

Đây là **nhánh tái hiện notebook EDA cũ**, không phải protocol reproduction chính của báo cáo: không One-Hot, không SMOTE, không StratifiedKFold. Để 13 mô hình đều chạy được, mỗi fold có adapter bắt buộc: impute `-999` fit trên training fold; các mô hình nhạy thang đo được StandardScaler fit trên training fold.

Notebook lưu checkpoint sau từng model/fold, OOF predictions, test predictions, metrics và submission để có thể tiếp tục khi Kaggle session bị ngắt.

## 1. Dependencies
Kaggle thường có sẵn các thư viện boosting. Cell này chỉ cài phần còn thiếu; bật Internet cho notebook nếu pip cần tải gói.

In [ ]:
%pip install -q -U lightgbm xgboost catboost pytorch-tabnet

## 2. Imports và cấu hình chạy
Mặc định chạy đủ 13 mô hình và 5 fold. Với KNN/GradientBoosting/TabNet, một Kaggle session có thể không đủ thời gian; có thể chia `MODELS_TO_RUN` hoặc `FOLDS_TO_RUN` qua nhiều session, giữ `RESUME=True`.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from pytorch_tabnet.classifier import TabNetClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, RandomForestClassifier)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, auc)
from sklearn.model_selection import KFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

SEED = 42
N_SPLITS = 5
DECISION_THRESHOLD = 0.5
PREDICTION_BATCH_SIZE = 8192
RESUME = True
SAVE_FOLD_MODELS = False
USE_GPU_IF_AVAILABLE = True
TABNET_MAX_EPOCHS = 100
TABNET_PATIENCE = 15

ALL_MODELS = [
    'Logistic_Regression', 'Decision_Tree', 'Random_Forest', 'LightGBM',
    'CatBoost', 'XGBoost', 'AdaBoost', 'Extra_Trees', 'KNN', 'LDA',
    'Naive_Bayes', 'Gradient_Boosting', 'TabNet',
]
MODELS_TO_RUN = ALL_MODELS.copy()
FOLDS_TO_RUN = list(range(1, N_SPLITS + 1))

OUTPUT_DIR = Path('/kaggle/working/ieee_legacy_13_models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert set(MODELS_TO_RUN).issubset(ALL_MODELS)
assert set(FOLDS_TO_RUN).issubset(range(1, N_SPLITS + 1))
print('GPU:', torch.cuda.is_available())
print('Models:', MODELS_TO_RUN)
print('Folds:', FOLDS_TO_RUN)
print('Output:', OUTPUT_DIR)

## 3. Tìm và đọc dữ liệu IEEE-CIS

In [ ]:
def resolve_competition_dir() -> Path:
    preferred = Path('/kaggle/input/ieee-fraud-detection')
    required = {'train_transaction.csv', 'train_identity.csv',
                'test_transaction.csv', 'test_identity.csv', 'sample_submission.csv'}
    if preferred.is_dir() and required.issubset({p.name for p in preferred.iterdir()}):
        return preferred
    candidates = []
    for path in Path('/kaggle/input').rglob('train_transaction.csv'):
        if required.issubset({p.name for p in path.parent.iterdir()}):
            candidates.append(path.parent)
    if len(candidates) != 1:
        raise FileNotFoundError(f'Không xác định được duy nhất thư mục IEEE-CIS: {candidates}')
    return candidates[0]

DATA_DIR = resolve_competition_dir()
print('Data directory:', DATA_DIR)

train_transaction = pd.read_csv(DATA_DIR / 'train_transaction.csv')
train_identity = pd.read_csv(DATA_DIR / 'train_identity.csv')
test_transaction = pd.read_csv(DATA_DIR / 'test_transaction.csv')
test_identity = pd.read_csv(DATA_DIR / 'test_identity.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left', validate='one_to_one')
test = test_transaction.merge(test_identity, on='TransactionID', how='left', validate='one_to_one')
assert len(train) == len(train_transaction)
assert len(test) == len(test_transaction)
assert train['TransactionID'].is_unique and test['TransactionID'].is_unique
print('Train merged:', train.shape)
print('Test merged :', test.shape)
print('Fraud distribution:')
display(train['isFraud'].value_counts().rename('count').to_frame().assign(rate=train['isFraud'].value_counts(normalize=True)))

del train_transaction, train_identity, test_transaction, test_identity
gc.collect()

## 4. EDA ngắn và feature engineering giống notebook cũ
Các group mean/std được tính riêng trên toàn train và toàn test, đúng với notebook tham khảo. Đây là lựa chọn transductive và group statistics của train có nhìn thấy validation; không dùng nhánh này để tuyên bố protocol chống leakage.

In [ ]:
print('Columns with missing values - train:', int(train.isna().any().sum()))
print('Columns with missing values - test :', int(test.isna().any().sum()))
print('Train TransactionDT:', int(train.TransactionDT.min()), '->', int(train.TransactionDT.max()))
print('Test TransactionDT :', int(test.TransactionDT.min()), '->', int(test.TransactionDT.max()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train['isFraud'].value_counts().sort_index().plot.bar(ax=axes[0], title='Class distribution')
axes[0].set_yscale('log')
axes[0].set_xlabel('isFraud')
axes[0].set_ylabel('Count (log scale)')
axes[1].hist(train['TransactionDT'], bins=80, alpha=.65, label='train')
axes[1].hist(test['TransactionDT'], bins=80, alpha=.65, label='test')
axes[1].set_title('TransactionDT train vs test')
axes[1].legend()
plt.tight_layout()
plt.show()

def add_legacy_group_features(df: pd.DataFrame) -> pd.DataFrame:
    specs = [
        ('TransactionAmt', 'card1'), ('TransactionAmt', 'card4'),
        ('id_02', 'card1'), ('id_02', 'card4'),
        ('D15', 'card1'), ('D15', 'card4'),
        ('D15', 'addr1'), ('D15', 'addr2'),
    ]
    for value_col, group_col in specs:
        if value_col not in df.columns or group_col not in df.columns:
            continue
        group = df.groupby(group_col, dropna=True)[value_col]
        df[f'{value_col}_to_mean_{group_col}'] = df[value_col] / group.transform('mean')
        df[f'{value_col}_to_std_{group_col}'] = df[value_col] / group.transform('std')
    for email_col in ('P_emaildomain', 'R_emaildomain'):
        if email_col in df.columns:
            parts = df[email_col].str.split('.', n=2, expand=True)
            for part_idx in range(3):
                df[f'{email_col}_{part_idx + 1}'] = parts[part_idx] if part_idx in parts.columns else np.nan
    return df

train = add_legacy_group_features(train)
test = add_legacy_group_features(test)
print('After feature engineering:', train.shape, test.shape)

## 5. Lọc cột và Label Encoding giống notebook cũ
Quy tắc: drop nếu missing >90%, top value >90%, hoặc chỉ có ≤1 giá trị trong train/test. Encoder được fit trên train+test để giữ đúng hành vi notebook cũ.

In [ ]:
def legacy_drop_candidates(df: pd.DataFrame) -> set[str]:
    many_null = set(df.columns[df.isna().mean() > 0.90])
    one_value = {c for c in df.columns if df[c].nunique(dropna=True) <= 1}
    dominant = set()
    for col in df.columns:
        frequencies = df[col].value_counts(dropna=False, normalize=True)
        if len(frequencies) and frequencies.iloc[0] > 0.90:
            dominant.add(col)
    return many_null | one_value | dominant

cols_to_drop = sorted(legacy_drop_candidates(train) | legacy_drop_candidates(test))
if 'isFraud' in cols_to_drop:
    cols_to_drop.remove('isFraud')
train.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print('Dropped columns:', len(cols_to_drop))
print('Shapes after drop:', train.shape, test.shape)

legacy_categorical_cols = [
    *[f'id_{i:02d}' for i in range(12, 39)], 'DeviceType', 'DeviceInfo',
    'ProductCD', 'card4', 'card6', 'M4', 'P_emaildomain', 'R_emaildomain',
    'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2',
    'M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9',
    'P_emaildomain_1', 'P_emaildomain_2', 'P_emaildomain_3',
    'R_emaildomain_1', 'R_emaildomain_2', 'R_emaildomain_3',
]
encoded_cols = []
for col in legacy_categorical_cols:
    if col not in train.columns or col not in test.columns:
        continue
    encoder = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0, ignore_index=True).astype(str)
    encoder.fit(combined)
    train[col] = encoder.transform(train[col].astype(str)).astype(np.int32)
    test[col] = encoder.transform(test[col].astype(str)).astype(np.int32)
    encoded_cols.append(col)
    del combined, encoder

train.replace([np.inf, -np.inf], np.nan, inplace=True)
test.replace([np.inf, -np.inf], np.nan, inplace=True)

drop_audit = pd.DataFrame({'dropped_column': cols_to_drop})
drop_audit.to_csv(OUTPUT_DIR / 'legacy_dropped_columns.csv', index=False)
pd.DataFrame({'encoded_column': encoded_cols}).to_csv(OUTPUT_DIR / 'legacy_encoded_columns.csv', index=False)
print('Encoded categorical columns:', len(encoded_cols))

## 6. Tạo X/y và 5 block KFold
Giống notebook cũ: train được sort theo thời gian, nhưng `TransactionDT` và `TransactionID` không đi vào model. `KFold(shuffle=False)` tạo validation block liên tiếp; đây không phải TimeSeriesSplit forward-only và cũng không stratified.

In [ ]:
train = train.sort_values('TransactionDT', kind='mergesort').reset_index(drop=True)
train_ids = train['TransactionID'].copy()
test_ids = test['TransactionID'].copy()
train_times = train['TransactionDT'].copy()

y = train['isFraud'].astype(np.int8).copy()
X = train.drop(columns=['isFraud', 'TransactionDT', 'TransactionID'])
X_test = test.drop(columns=['TransactionDT', 'TransactionID'])
assert list(X.columns) == list(X_test.columns)
assert X.select_dtypes(exclude=[np.number]).empty

fold_indices = list(KFold(n_splits=N_SPLITS, shuffle=False).split(X))
fold_summary = []
for fold, (train_idx, valid_idx) in enumerate(fold_indices, 1):
    fold_summary.append({
        'fold': fold, 'train_rows': len(train_idx), 'valid_rows': len(valid_idx),
        'train_fraud_rate': float(y.iloc[train_idx].mean()),
        'valid_fraud_rate': float(y.iloc[valid_idx].mean()),
        'valid_time_min': int(train_times.iloc[valid_idx].min()),
        'valid_time_max': int(train_times.iloc[valid_idx].max()),
    })
fold_summary = pd.DataFrame(fold_summary)
fold_summary.to_csv(OUTPUT_DIR / 'fold_summary.csv', index=False)
display(fold_summary)
print('X:', X.shape, '| X_test:', X_test.shape)

del train, test
gc.collect()

## 7. Registry đủ 12 baseline + TabNet
Các tham số đã nêu rõ trong báo cáo được ưu tiên: RF 100 cây; XGBoost/LightGBM/CatBoost learning rate 0.1 và 200 estimators; TabNet `n_d=n_a=64`, Adam `lr=0.02`, batch 1024, tối đa 100 epoch. Các tham số báo cáo không nêu được ghi nhận là implementation choices.

In [ ]:
GPU_AVAILABLE = USE_GPU_IF_AVAILABLE and torch.cuda.is_available()
SCALE_MODELS = {'Logistic_Regression', 'KNN', 'LDA', 'Naive_Bayes', 'TabNet'}

def build_model(name: str, seed: int):
    models = {
        'Logistic_Regression': lambda: LogisticRegression(max_iter=1000, random_state=seed, n_jobs=-1),
        'Decision_Tree': lambda: DecisionTreeClassifier(max_depth=10, random_state=seed),
        'Random_Forest': lambda: RandomForestClassifier(n_estimators=100, max_depth=None, random_state=seed, n_jobs=-1),
        'LightGBM': lambda: LGBMClassifier(n_estimators=200, learning_rate=0.1, objective='binary', random_state=seed, n_jobs=-1, verbosity=-1),
        'CatBoost': lambda: CatBoostClassifier(iterations=200, learning_rate=0.1, loss_function='Logloss', eval_metric='AUC', random_seed=seed, verbose=False, task_type='GPU' if GPU_AVAILABLE else 'CPU'),
        'XGBoost': lambda: XGBClassifier(n_estimators=200, learning_rate=0.1, objective='binary:logistic', eval_metric='logloss', tree_method='hist', device='cuda' if GPU_AVAILABLE else 'cpu', random_state=seed, n_jobs=-1),
        'AdaBoost': lambda: AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=seed),
        'Extra_Trees': lambda: ExtraTreesClassifier(n_estimators=100, max_depth=None, random_state=seed, n_jobs=-1),
        'KNN': lambda: KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'LDA': lambda: LinearDiscriminantAnalysis(),
        'Naive_Bayes': lambda: GaussianNB(),
        'Gradient_Boosting': lambda: GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=seed),
        'TabNet': lambda: TabNetClassifier(
            n_d=64, n_a=64, n_steps=5, gamma=1.5, lambda_sparse=1e-3,
            optimizer_fn=torch.optim.Adam, optimizer_params={'lr': 2e-2},
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            scheduler_params={'step_size': 10, 'gamma': 0.9},
            mask_type='sparsemax', seed=seed, verbose=10,
        ),
    }
    if name not in models:
        raise KeyError(name)
    return models[name]()

assert len(ALL_MODELS) == 13 and len(set(ALL_MODELS)) == 13
display(pd.DataFrame({'model': ALL_MODELS, 'scale_after_impute': [m in SCALE_MODELS for m in ALL_MODELS]}))

## 8. Fold-local compatibility adapter, training và metrics
Imputer/scaler chỉ fit trên training fold. Không SMOTE để giữ đúng nhánh EDA cũ. Test probability được dự đoán theo batch nhằm giảm đỉnh RAM.

In [ ]:
def prepare_fold(model_name, train_idx, valid_idx):
    imputer = SimpleImputer(strategy='constant', fill_value=-999.0)
    X_train_arr = imputer.fit_transform(X.iloc[train_idx]).astype(np.float32, copy=False)
    X_valid_arr = imputer.transform(X.iloc[valid_idx]).astype(np.float32, copy=False)
    X_test_arr = imputer.transform(X_test).astype(np.float32, copy=False)
    scaler = None
    if model_name in SCALE_MODELS:
        scaler = StandardScaler()
        X_train_arr = scaler.fit_transform(X_train_arr).astype(np.float32, copy=False)
        X_valid_arr = scaler.transform(X_valid_arr).astype(np.float32, copy=False)
        X_test_arr = scaler.transform(X_test_arr).astype(np.float32, copy=False)
    return X_train_arr, X_valid_arr, X_test_arr, imputer, scaler

def fit_model(model_name, model, X_train_arr, y_train_arr, X_valid_arr, y_valid_arr):
    if model_name == 'TabNet':
        model.fit(
            X_train_arr, y_train_arr, eval_set=[(X_valid_arr, y_valid_arr)],
            eval_name=['validation'], eval_metric=['auc'],
            max_epochs=TABNET_MAX_EPOCHS, patience=TABNET_PATIENCE,
            batch_size=1024, virtual_batch_size=128, num_workers=2, drop_last=False,
        )
    else:
        model.fit(X_train_arr, y_train_arr)
    return model

def predict_proba_batched(model, X_arr, batch_size=PREDICTION_BATCH_SIZE):
    parts = []
    for start in range(0, len(X_arr), batch_size):
        parts.append(model.predict_proba(X_arr[start:start + batch_size])[:, 1])
    return np.concatenate(parts).astype(np.float32, copy=False)

def classification_metrics(y_true, scores, threshold=DECISION_THRESHOLD):
    predictions = (scores >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    curve_precision, curve_recall, _ = precision_recall_curve(y_true, scores)
    return {
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'Precision': precision_score(y_true, predictions, zero_division=0),
        'Recall': recall_score(y_true, predictions, zero_division=0),
        'F1_Score': f1_score(y_true, predictions, zero_division=0),
        'ROC_AUC': roc_auc_score(y_true, scores),
        'PR_AUC': auc(curve_recall, curve_precision),
        'Average_Precision': average_precision_score(y_true, scores),
    }

def save_model_if_requested(model_name, model, model_path):
    if not SAVE_FOLD_MODELS:
        return
    if model_name == 'TabNet':
        model.save_model(str(model_path))
    else:
        joblib.dump(model, model_path.with_suffix('.joblib'))

In [ ]:
all_fold_metrics = []
model_oof = {}
model_test_predictions = {}

for model_name in MODELS_TO_RUN:
    print(f'\n===== {model_name} =====')
    model_dir = OUTPUT_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    oof_scores = np.full(len(y), np.nan, dtype=np.float32)
    test_sum = np.zeros(len(X_test), dtype=np.float64)
    completed_folds = 0

    for fold, (train_idx, valid_idx) in enumerate(fold_indices, 1):
        if fold not in FOLDS_TO_RUN:
            continue
        pred_path = model_dir / f'fold_{fold:02d}_predictions.npz'
        metric_path = model_dir / f'fold_{fold:02d}_metrics.json'

        if RESUME and pred_path.exists() and metric_path.exists():
            saved = np.load(pred_path)
            valid_scores = saved['valid_scores']
            test_scores = saved['test_scores']
            with metric_path.open('r', encoding='utf-8') as handle:
                fold_metrics = json.load(handle)
            print(f'Fold {fold}: loaded checkpoint')
        else:
            started = time.perf_counter()
            X_train_arr, X_valid_arr, X_test_arr, imputer, scaler = prepare_fold(model_name, train_idx, valid_idx)
            y_train_arr = y.iloc[train_idx].to_numpy(dtype=np.int64)
            y_valid_arr = y.iloc[valid_idx].to_numpy(dtype=np.int64)
            model = build_model(model_name, SEED + fold)
            model = fit_model(model_name, model, X_train_arr, y_train_arr, X_valid_arr, y_valid_arr)
            training_seconds = time.perf_counter() - started

            prediction_started = time.perf_counter()
            valid_scores = predict_proba_batched(model, X_valid_arr)
            test_scores = predict_proba_batched(model, X_test_arr)
            prediction_seconds = time.perf_counter() - prediction_started
            fold_metrics = {
                'model': model_name, 'fold': fold, 'seed': SEED + fold,
                'train_rows': len(train_idx), 'valid_rows': len(valid_idx),
                'train_fraud_rate': float(y_train_arr.mean()),
                'valid_fraud_rate': float(y_valid_arr.mean()),
                'training_seconds': training_seconds,
                'prediction_seconds': prediction_seconds,
                **classification_metrics(y_valid_arr, valid_scores),
            }
            np.savez_compressed(pred_path, valid_idx=valid_idx, valid_scores=valid_scores, test_scores=test_scores)
            with metric_path.open('w', encoding='utf-8') as handle:
                json.dump(fold_metrics, handle, ensure_ascii=False, indent=2)
            save_model_if_requested(model_name, model, model_dir / f'fold_{fold:02d}_model')
            del model, X_train_arr, X_valid_arr, X_test_arr, y_train_arr, y_valid_arr, imputer, scaler
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        oof_scores[valid_idx] = valid_scores
        test_sum += test_scores
        completed_folds += 1
        all_fold_metrics.append(fold_metrics)
        print(f"Fold {fold}: F1={fold_metrics['F1_Score']:.6f} | ROC-AUC={fold_metrics['ROC_AUC']:.6f} | PR-AUC={fold_metrics['PR_AUC']:.6f}")

    if completed_folds:
        valid_mask = ~np.isnan(oof_scores)
        aggregate = {'model': model_name, 'fold': 'OOF', 'completed_folds': completed_folds,
                     **classification_metrics(y.to_numpy()[valid_mask], oof_scores[valid_mask])}
        with (model_dir / 'oof_metrics.json').open('w', encoding='utf-8') as handle:
            json.dump(aggregate, handle, ensure_ascii=False, indent=2)
        pd.DataFrame({'TransactionID': train_ids[valid_mask].to_numpy(),
                      'isFraud_true': y.to_numpy()[valid_mask],
                      'isFraud_score': oof_scores[valid_mask]}).to_csv(model_dir / 'oof_predictions.csv', index=False)
        model_oof[model_name] = aggregate
        model_test_predictions[model_name] = (test_sum / completed_folds).astype(np.float32)
        print('OOF:', aggregate)

pd.DataFrame(all_fold_metrics).to_csv(OUTPUT_DIR / 'all_fold_metrics.csv', index=False)
print('Training loop complete.')

## 9. Bảng so sánh, biểu đồ và Kaggle submissions

In [ ]:
if model_oof:
    comparison = pd.DataFrame(model_oof.values()).sort_values('F1_Score', ascending=False)
    comparison.to_csv(OUTPUT_DIR / 'model_comparison_oof.csv', index=False)
    display(comparison[['model', 'completed_folds', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC', 'PR_AUC', 'Average_Precision']])

    plot_data = comparison.melt(id_vars='model', value_vars=['F1_Score', 'ROC_AUC', 'PR_AUC'],
                                var_name='metric', value_name='score')
    plt.figure(figsize=(14, 6))
    sns.barplot(data=plot_data, x='model', y='score', hue='metric')
    plt.xticks(rotation=55, ha='right')
    plt.ylim(0, 1)
    plt.title('IEEE-CIS legacy EDA pipeline — OOF comparison')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'model_comparison_oof.png', dpi=180, bbox_inches='tight')
    plt.show()

for model_name, test_scores in model_test_predictions.items():
    submission = sample_submission[['TransactionID']].copy()
    if not np.array_equal(submission['TransactionID'].to_numpy(), test_ids.to_numpy()):
        score_map = pd.Series(test_scores, index=test_ids.to_numpy())
        submission['isFraud'] = submission['TransactionID'].map(score_map)
    else:
        submission['isFraud'] = test_scores
    assert submission['isFraud'].notna().all()
    submission.to_csv(OUTPUT_DIR / f'submission_{model_name}.csv', index=False)

print('Artifacts:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), f'({path.stat().st_size / 1024**2:.2f} MB)')

## 10. Cách diễn giải kết quả

- So sánh nghiên cứu chính dùng `Precision`, `Recall`, `F1_Score`, `ROC_AUC`, `PR_AUC`; Accuracy không được dùng làm kết luận.
- Chỉ so sánh các model khi `completed_folds` giống nhau. Nếu chia model/fold qua nhiều session, tải lại checkpoint vào cùng output dataset hoặc chạy lại notebook với output của session trước được gắn làm input.
- Submission chỉ phục vụ Kaggle leaderboard vì competition test không có ground truth công khai. Các metrics trong notebook là OOF trên train có nhãn.
- Pipeline này cố ý giữ Label Encoding, lọc cột từ train+test, group statistics toàn tập và blocked KFold của notebook EDA cũ. Không trộn bảng kết quả này với bảng reproduction dùng One-Hot + StandardScaler + SMOTE + StratifiedKFold.
- XAI chưa nằm trong notebook này. SHAP/LIME/DiCE và giải thích nội tại TabNet chỉ nên chạy sau khi đã chọn và khóa model/config mục tiêu.